Mini API ETL Pipeline
=====================

API → Extract → Transform → Validate → Log → MySQL

Tools:
- Python
- Requests
- Pandas
- MySQL
- Logging

**EXTRACTION**

**Step 1 — Choose our API**
For learning, we'll use JSONPlaceholder, a free fake REST API designed for testing and learning.

Our first endpoint will be:
https://jsonplaceholder.typicode.com/users

**Step 2 — Import libraries**

In [59]:
import requests
import pandas as pd
import logging 
import time
#Don't add MySQL yet. We'll introduce it when we reach the Load stage.

**Task 1**
Write the code that:
Stores the JSONPlaceholder users URL in a variable called url
Sends a GET request using requests
Uses a 10-second timeout
Checks whether the request was successful
Converts the response into Python data

In [60]:
url="https://jsonplaceholder.typicode.com/users"
response=requests.get(url,timeout=10)
response.raise_for_status()
data=response.json()
df=pd.DataFrame(data)

#Now inspect what we extracted
print(response.status_code)    #200 → API request successful
print(len(data)) #10 → 10 records extracted
print(df.head())
print(df.shape) #(10, 8) → 10 rows × 8 columns
print()

print(data[0])
print()

print(data[0]["address"])


# Order:
# requests.get()
#       ↓
# Did API request succeed?
#       ↓
# raise_for_status()
#       ↓
# Convert JSON
#       ↓
# DataFrame

200
10
   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \
0  {'street': 'Kulas Light', 'suite': 'Apt. 556',...  1-770-736-8031 x56442   
1  {'street': 'Victor Plains', 'suite': 'Suite 87...    010-692-6593 x09125   
2  {'street': 'Douglas Extension', 'suite': 'Suit...         1-463-123-4447   
3  {'street': 'Hoeger Mall', 'suite': 'Apt. 692',...      493-170-9623 x156   
4  {'street': 'Skiles Walks', 'suite': 'Suite 351...          (254)954-1289   

         website                                            company  
0  hildegard.org  {'name': 'Romaguera-Crona', 'catchPhras

**TRANSFORMATION**

**Step 3 — Decide what our ETL table should contain**

We don't necessarily want to dump the raw API response directly into MySQL.
Because, MySQL relational tables are designed around rows and columns, while the API response can contain nested dictionaries, lists, optional fields, etc.Nested JSON doesn't fit naturally into columns.

In a more accurate statement:
**We don't usually load raw API data directly into the final relational/analytics table. We may preserve the raw data in a staging/raw layer, then transform and validate it before loading the structured data used downstream.**

Let's create a clean customer table:
id
name
username
email
city
zipcode
latitude
longitude
company_name

So we're going from:
Raw API JSON
     ↓
Nested dictionaries
     ↓
Clean tabular data
     ↓
Database

**This is our Transform stage.**

In [61]:
#First transformation: flatten the nested JSON 
#using json_normalize

#json_normalize() transforms nested, semi-structured JSON data into a flat,
#two-dimensional table (a Pandas or Polars DataFrame).

df_flat=pd.json_normalize(data)
print(df_flat.head())
print(df_flat.columns)


   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburgh

**Transaformation step 1: Select Required columns**

In [62]:
# Now let's select only the columns our customer table actually needs.
# From your output, we'll keep:
# id
# name
# username
# email
# address.city
# address.zipcode
# address.geo.lat
# address.geo.lng
# company.name

selected_cols=df_flat[["id","name","username","email","address.city","address.zipcode",
              "address.geo.lat","address.geo.lng","company.name"]]
#Now lets inspect selected columns

print(selected_cols.head()) 
print(selected_cols.shape)
print()



   id              name   username                      email   address.city  \
0   1     Leanne Graham       Bret          Sincere@april.biz    Gwenborough   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv    Wisokyburgh   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net  McKenziehaven   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org    South Elvis   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca     Roscoeview   

  address.zipcode address.geo.lat address.geo.lng        company.name  
0      92998-3874        -37.3159         81.1496     Romaguera-Crona  
1      90566-7771        -43.9509        -34.4618        Deckow-Crist  
2      59590-4157        -68.6102        -47.0653  Romaguera-Jacobson  
3      53919-4257         29.4572       -164.2990       Robel-Corkery  
4           33263        -31.8129         62.5342         Keebler LLC  
(10, 9)



**Transaformation step 2: Rename columns**

Right now we have names like:
address.city
address.geo.lat
company.name

These work in Pandas, but for a database table, cleaner names would be:
id
name
username
email
city
zipcode
latitude
longitude
company_name

In [63]:
selected_cols=selected_cols.rename(columns={
           "address.city":"city",
           "address.zipcode":"zipcode",
           "address.geo.lat":"latitude",
           "address.geo.lng":"longitude",
           "company.name":"company_name"
            })
print(selected_cols.head())
print(selected_cols.columns)

   id              name   username                      email           city  \
0   1     Leanne Graham       Bret          Sincere@april.biz    Gwenborough   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv    Wisokyburgh   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net  McKenziehaven   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org    South Elvis   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca     Roscoeview   

      zipcode  latitude  longitude        company_name  
0  92998-3874  -37.3159    81.1496     Romaguera-Crona  
1  90566-7771  -43.9509   -34.4618        Deckow-Crist  
2  59590-4157  -68.6102   -47.0653  Romaguera-Jacobson  
3  53919-4257   29.4572  -164.2990       Robel-Corkery  
4       33263  -31.8129    62.5342         Keebler LLC  
Index(['id', 'name', 'username', 'email', 'city', 'zipcode', 'latitude',
       'longitude', 'company_name'],
      dtype='object')


**Transaformation step 3: Data Type Validation & Transformation**

In [64]:
print(selected_cols.dtypes)

id               int64
name            object
username        object
email           object
city            object
zipcode         object
latitude        object
longitude       object
company_name    object
dtype: object


In [65]:
#Why are latitude/longitude object?
#Because the API gave them to us as strings:
#For analytics/database use, we want numeric values:

selected_cols["latitude"]=selected_cols["latitude"].astype(float)
selected_cols["longitude"]=selected_cols["longitude"].astype(float)
print(selected_cols.dtypes)



id                int64
name             object
username         object
email            object
city             object
zipcode          object
latitude        float64
longitude       float64
company_name     object
dtype: object


**Transaformation step 4: Data Validation**

Before sending anything to MySQL, we need to ask:
"Is this data actually valid?"

or our customer data, let's define some simple validation rules:
Rule 1 — id must be unique
Rule 2 — Required fields cannot be missing(id,name,email)
Rule 3 — Latitude must be between -90 and 90
Rule 4 — Longitude must be between -180 and 180

In [66]:
#Rule 1 — id must be unique
print(selected_cols["id"].duplicated())
print(selected_cols["id"].duplicated().sum()) #0,that means there are no duplicate IDs.


0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: id, dtype: bool
0


In [67]:
#Rule 2 — Required fields cannot be missing(id,name,email)
print(selected_cols[["id","name","email"]].isnull())
print(selected_cols[["id","name","email"]].isnull().sum())


      id   name  email
0  False  False  False
1  False  False  False
2  False  False  False
3  False  False  False
4  False  False  False
5  False  False  False
6  False  False  False
7  False  False  False
8  False  False  False
9  False  False  False
id       0
name     0
email    0
dtype: int64


In [68]:
#Rule 3 — Latitude must be between -90 and 90
invalid_latitude=selected_cols[
    (selected_cols["latitude"]<-90)|(selected_cols["latitude"]>90)
    ]
print(invalid_latitude)
print()
print(len(invalid_latitude))

Empty DataFrame
Columns: [id, name, username, email, city, zipcode, latitude, longitude, company_name]
Index: []

0


In [69]:
#Rule 4 — Longitude must be between -180 and 180
invalid_longitude=selected_cols[
    (selected_cols["longitude"]<-180) 
     |
    (selected_cols["longitude"]>180)
     ]

print(invalid_longitude)
print(len(invalid_longitude))

Empty DataFrame
Columns: [id, name, username, email, city, zipcode, latitude, longitude, company_name]
Index: []
0


**Transaformation step 5: Logging**
Now we're going to make the pipeline record what happened.
For example, instead of only:
print("Validation passed")

we'll have a log file like:
2026-09-21 11:30:10 - INFO - API request successful
2026-09-21 11:30:10 - INFO - 10 records extracted
2026-09-21 11:30:11 - INFO - Data validation started
2026-09-21 11:30:11 - INFO - Validation passed

In [70]:
import logging
logging.basicConfig(
    filename="pipeline.log", #Store my log messages inside a file called pipeline.log.
    level=logging.INFO,      #his tells Python the minimum level of messages to record.
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
    
)

#You already calculated:
#selected_cols["id"].duplicated().sum()
#Instead of only printing it, we can store it:

duplicated_ids=selected_cols["id"].duplicated().sum()

#Then log to Rule 1:
logging.info(f"Duplicate ID's:{duplicated_ids}")


#Now let's log Rule 2
missing_values = selected_cols[["id", "name", "email"]].isnull().sum().sum()
logging.info(f"Missing required values's:{missing_values}")

#log to Rule 3
logging.info(f"Invalid latitude:{len(invalid_latitude)}")

#log to Rule 4
logging.info(f"Invalid longitude:{len(invalid_longitude)}")

In [71]:
#Check whether the file exists
import os

print(os.path.exists("pipeline.log"))

True


In [72]:
#To see logging info in pipeline.log file
with open("pipeline.log", "r") as file:
    logs = file.read()

print(logs)

2026-09-21 11:43:18,975 - INFO - Data validation started
2026-09-21 11:44:25,728 - INFO - Data validation started
2026-09-21 11:44:27,265 - INFO - Data validation started
2026-09-21 11:44:31,109 - INFO - Data validation started
2026-09-21 11:47:40,255 - INFO - Duplicate ID's:0
2026-09-21 11:47:49,801 - INFO - Duplicate ID's:0
2026-09-21 11:47:55,863 - INFO - Duplicate ID's:0
2026-09-21 11:49:21,393 - INFO - Duplicate ID's:0
2026-09-21 12:04:08,161 - INFO - Duplicate ID's:0
2026-09-21 12:04:08,168 - INFO - Duplicate ID's:0
2026-09-21 12:04:43,686 - INFO - Duplicate ID's:0
2026-09-21 12:04:43,688 - INFO - Missing required values's:0
2026-09-21 12:06:19,333 - INFO - Duplicate ID's:0
2026-09-21 12:06:19,335 - INFO - Missing required values's:0
2026-09-21 12:06:19,339 - INFO - Invalid latitude:Empty DataFrame
Columns: [id, name, username, email, city, zipcode, latitude, longitude, company_name]
Index: []
2026-09-21 12:06:55,743 - INFO - Duplicate ID's:0
2026-09-21 12:06:55,743 - INFO - Miss

**Transaformation step 6: validation + Logging**
Right now we're logging each individual rule.
But a real pipeline should also have an overall validation result.
We want the logic:
If:
    duplicate IDs = 0
    AND missing values = 0
    AND invalid latitude = 0
    AND invalid longitude = 0

Then:
    Validation PASSED

Otherwise:
    Validation FAILED

In [73]:
if (duplicated_ids==0 and
    missing_values ==0 and 
    len(invalid_latitude)==0 and #we needed the number of invalid records so we use len()
    len(invalid_longitude)==0):  #we needed the number of invalid records so we use len()
    logging.info("Data Validation Passed")
else:
    logging.error("Data Validation Failed")

In [74]:
#To see logging info in pipeline.log file
with open("pipeline.log", "r") as file:
    logs = file.read()

print(logs)

2026-09-21 11:43:18,975 - INFO - Data validation started
2026-09-21 11:44:25,728 - INFO - Data validation started
2026-09-21 11:44:27,265 - INFO - Data validation started
2026-09-21 11:44:31,109 - INFO - Data validation started
2026-09-21 11:47:40,255 - INFO - Duplicate ID's:0
2026-09-21 11:47:49,801 - INFO - Duplicate ID's:0
2026-09-21 11:47:55,863 - INFO - Duplicate ID's:0
2026-09-21 11:49:21,393 - INFO - Duplicate ID's:0
2026-09-21 12:04:08,161 - INFO - Duplicate ID's:0
2026-09-21 12:04:08,168 - INFO - Duplicate ID's:0
2026-09-21 12:04:43,686 - INFO - Duplicate ID's:0
2026-09-21 12:04:43,688 - INFO - Missing required values's:0
2026-09-21 12:06:19,333 - INFO - Duplicate ID's:0
2026-09-21 12:06:19,335 - INFO - Missing required values's:0
2026-09-21 12:06:19,339 - INFO - Invalid latitude:Empty DataFrame
Columns: [id, name, username, email, city, zipcode, latitude, longitude, company_name]
Index: []
2026-09-21 12:06:55,743 - INFO - Duplicate ID's:0
2026-09-21 12:06:55,743 - INFO - Miss

**LOADING**

Load → MySQL
We'll take our clean selected_cols DataFrame and:
Connect Python → MySQL
Create a database/table
Insert the cleaned records
Commit the transaction
Log whether the load succeeded or failed
Query MySQL to verify the data

**Step 1: Connect Python to MySQL**

In [75]:
import sys

print(sys.executable)

C:\Users\K Vaishnavi\AppData\Local\Programs\Python\Python312\python.exe


In [76]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "mysql-connector-python"
])

0

In [77]:
import mysql.connector

print("MySQL connector imported successfully!")

MySQL connector imported successfully!


In [78]:
#Import the connector
import mysql.connector

#Create the connection
connection=mysql.connector.connect(
    host="localhost", #Where MySQL is running
    user="root", #Your MySQL username
    password="your_actual_password" #Your MySQL password
) 

#Test the connection
print(connection.is_connected())


True


**Step 2: Create the database**
Our API ETL project needs its own database. Let's call it:
api_etl_db

In [79]:
# Create cursor
cursor = connection.cursor()

# Create database
cursor.execute("CREATE DATABASE IF NOT EXISTS api_etl_db")
print("Database created successfully!")
print()

# Verify database
cursor.execute("SHOW DATABASES")
print("\nAvailable databases:")

for database in cursor:
    print(database[0])

Database created successfully!


Available databases:
api_etl_db
information_schema
mysql
performance_schema
sys


**Step 3: Create the customers table**
Our selected_cols currently contains:
id
name
username
email
city
zipcode
latitude
longitude
company_name

We need to create matching columns in MySQL.

In [80]:
# Connect to our project database
connection=mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_actual_password",
    database="api_etl_db"
)

cursor=connection.cursor()

# Create customers table
create_table_query="""
CREATE TABLE IF NOT EXISTS customers(
      id INT primary key,
      name varchar(100),
      username varchar(100),
      email varchar(150),
      city varchar(100),
      zipcode varchar(20),
      latitude float,
      longitude float,
      company_name varchar(150)
)
"""

cursor.execute(create_table_query)
print("Customer table created successfully!")

Customer table created successfully!


**Step 4: Load selected_cols into MySQL**
Now we're going to take the 9 rows/columns from your Pandas DataFrame and insert them into the customers table.

Your DataFrame has:
id
name
username
email
city
zipcode
latitude
longitude
company_name

In [81]:
# SQL query for inserting one customer
insert_query = """
INSERT INTO customers
(id, name, username, email, city, zipcode, latitude, longitude, company_name)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

# Insert each row from the DataFrame
for _, row in selected_cols.iterrows():

    values = (
        row["id"],
        row["name"],
        row["username"],
        row["email"],
        row["city"],
        row["zipcode"],
        row["latitude"],
        row["longitude"],
        row["company_name"]
    )

    cursor.execute(insert_query, values)

# Save changes
connection.commit() #Save all the changes I just made to the database

print("Data loaded successfully!")

#WE may get error if we run more than once because id is a primary key, inserting this more than once fails.

IntegrityError: 1062 (23000): Duplicate entry '1' for key 'customers.PRIMARY'

In [82]:
cursor.execute("SELECT COUNT(*) FROM customers")
count = cursor.fetchone()[0]
print("Number of records:", count)

Number of records: 10


**Step 5: Verify the loaded data**
Now let's actually retrieve the records from MySQL, rather than trusting the DataFrame.

In [83]:
cursor.execute(""" 
select * 
from customers
""")

rows=cursor.fetchall()

for row in rows:
    print(row)

(1, 'Leanne Graham', 'Bret', 'Sincere@april.biz', 'Gwenborough', '92998-3874', -37.3159, 81.1496, 'Romaguera-Crona')
(2, 'Ervin Howell', 'Antonette', 'Shanna@melissa.tv', 'Wisokyburgh', '90566-7771', -43.9509, -34.4618, 'Deckow-Crist')
(3, 'Clementine Bauch', 'Samantha', 'Nathan@yesenia.net', 'McKenziehaven', '59590-4157', -68.6102, -47.0653, 'Romaguera-Jacobson')
(4, 'Patricia Lebsack', 'Karianne', 'Julianne.OConner@kory.org', 'South Elvis', '53919-4257', 29.4572, -164.299, 'Robel-Corkery')
(5, 'Chelsey Dietrich', 'Kamren', 'Lucio_Hettinger@annie.ca', 'Roscoeview', '33263', -31.8129, 62.5342, 'Keebler LLC')
(6, 'Mrs. Dennis Schulist', 'Leopoldo_Corkery', 'Karley_Dach@jasper.info', 'South Christy', '23505-1337', -71.4197, 71.7478, 'Considine-Lockman')
(7, 'Kurtis Weissnat', 'Elwyn.Skiles', 'Telly.Hoeger@billy.biz', 'Howemouth', '58804-1099', 24.8918, 21.8984, 'Johns Group')
(8, 'Nicholas Runolfsdottir V', 'Maxime_Nienow', 'Sherwood@rosamond.me', 'Aliyaview', '45169', -14.399, -120.768,

In [84]:
cursor.execute("""
SELECT city, COUNT(*) AS customer_count
FROM customers
GROUP BY city
""")

results = cursor.fetchall()

for row in results:
    print(row)

('Gwenborough', 1)
('Wisokyburgh', 1)
('McKenziehaven', 1)
('South Elvis', 1)
('Roscoeview', 1)
('South Christy', 1)
('Howemouth', 1)
('Aliyaview', 1)
('Bartholomebury', 1)
('Lebsackbury', 1)


**Load Mysql connection logging details to pipeline.log file**

In [85]:
logging.info(f"Successfully loaded {len(selected_cols)} records into customers table")

In [86]:
with open("pipeline.log", "r") as file:
    print(file.read())

2026-09-21 11:43:18,975 - INFO - Data validation started
2026-09-21 11:44:25,728 - INFO - Data validation started
2026-09-21 11:44:27,265 - INFO - Data validation started
2026-09-21 11:44:31,109 - INFO - Data validation started
2026-09-21 11:47:40,255 - INFO - Duplicate ID's:0
2026-09-21 11:47:49,801 - INFO - Duplicate ID's:0
2026-09-21 11:47:55,863 - INFO - Duplicate ID's:0
2026-09-21 11:49:21,393 - INFO - Duplicate ID's:0
2026-09-21 12:04:08,161 - INFO - Duplicate ID's:0
2026-09-21 12:04:08,168 - INFO - Duplicate ID's:0
2026-09-21 12:04:43,686 - INFO - Duplicate ID's:0
2026-09-21 12:04:43,688 - INFO - Missing required values's:0
2026-09-21 12:06:19,333 - INFO - Duplicate ID's:0
2026-09-21 12:06:19,335 - INFO - Missing required values's:0
2026-09-21 12:06:19,339 - INFO - Invalid latitude:Empty DataFrame
Columns: [id, name, username, email, city, zipcode, latitude, longitude, company_name]
Index: []
2026-09-21 12:06:55,743 - INFO - Duplicate ID's:0
2026-09-21 12:06:55,743 - INFO - Miss

**Your project is essentially complete**
1. API Extraction       ✅
2. JSON Normalization   ✅
3. Data Transformation  ✅
4. Data Type Conversion ✅
5. Data Validation      ✅
6. Logging              ✅
7. MySQL Database       ✅
8. MySQL Table          ✅
9. Data Loading         ✅
10. SQL Verification    ✅